In [2]:
import pandas as pd

file_path = 'user_cognitive_metrics.csv'

# Read the CSV file into a DataFrame
data = pd.read_csv(file_path)


data = data.rename(columns={
    'user_id_column_name': 'user_id',
    'duration_column_name': 'Duration',
    'visiting_frequency_column_name': 'Visiting Frequency',
    'product_category_consistency_column_name': 'Product Category Consistency',
    'brand_consistency_column_name': 'Brand Consistency'
})

# Ensuring all necessary columns are present
required_columns = ['user_id', 'Duration', 'Visiting Frequency', 'Product Category Consistency', 'Brand Consistency']
if not all(column in data.columns for column in required_columns):
    raise ValueError("The DataFrame does not contain all the required columns.")

# Calculating thresholds based on the 75th percentile
thresholds = {
    'Duration': data['Duration'].quantile(0.75),
    'Visiting Frequency': data['Visiting Frequency'].quantile(0.75),
    'Product Category Consistency': data['Product Category Consistency'].quantile(0.75),
    'Brand Consistency': data['Brand Consistency'].quantile(0.75)
}

# Calculating Combined Consistency (CC) and assign scores based on thresholds
data['CC'] = (data['Product Category Consistency'] + data['Brand Consistency']) / 2
data['Score_Duration'] = (data['Duration'] > thresholds['Duration']).astype(int)
data['Score_CC'] = (data['CC'] < thresholds['Product Category Consistency']).astype(int)
data['Score_VF'] = (data['Visiting Frequency'] > thresholds['Visiting Frequency']).astype(int)

# Aggregate the scores to determine the cognitive style
data['Aggregate_Score'] = data['Score_Duration'] + data['Score_CC'] + data['Score_VF']
data['Cognitive_Style'] = data['Aggregate_Score'].apply(lambda x: 'Abstract' if x >= 2 else 'Concrete')

# Selecting only the required columns to output
output_data = data[['user_id', 'Cognitive_Style']]

# Saving the output DataFrame to a new CSV file
output_data.to_csv('user_cognitive_styles.csv', index=False)
print("Cognitive style classifications saved to 'user_cognitive_styles.csv'.")

# Print out a preview of the output DataFrame
print(output_data.head())


Cognitive style classifications saved to 'user_cognitive_styles.csv'.
       user_id Cognitive_Style
0  274969076.0        Concrete
1  275256741.0        Concrete
2  295643776.0        Concrete
3  296465302.0        Concrete
4  356520186.0        Concrete


In [3]:
# Counting the number of occurrences of each Cognitive Style
style_counts = data['Cognitive_Style'].value_counts()

# Printing the counts
print(style_counts)

Concrete    66171
Abstract    33829
Name: Cognitive_Style, dtype: int64


In [5]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

file_path = 'user_cognitive_metrics.csv'
data = pd.read_csv(file_path)

# Selecting relevant features for clustering
features = data[['Duration', 'Visiting Frequency', 'Product Category Consistency', 'Brand Consistency']]

# Handling missing values by imputing with the mean
imputer = SimpleImputer(missing_values=np.nan, strategy='mean')
features_imputed = imputer.fit_transform(features)

# Scaling the features
scaler = StandardScaler()
features_scaled = scaler.fit_transform(features_imputed)

# Initializing and fit the KMeans model
kmeans = KMeans(n_clusters=2, random_state=0)
kmeans.fit(features_scaled)

# Predicting the clusters
predicted_clusters = kmeans.predict(features_scaled)

# Add the predicted clusters to the original dataframe
data['Cognitive_Style'] = predicted_clusters

# Select only the required columns to output
output_data = data[['user_id', 'Cognitive_Style']]

# Save the output DataFrame to a new CSV file
output_data.to_csv('KMeans Clusters.csv', index=False)
print("Clusters are saved to  'KMeans Clusters.csv'.")

# Print out a preview of the output DataFrame
print(output_data.head())


Clusters are saved to  'KMeans Clusters.csv'.
       user_id  Cognitive_Style
0  274969076.0                0
1  275256741.0                0
2  295643776.0                0
3  296465302.0                0
4  356520186.0                1


In [6]:
import pandas as pd

# Loading both datasets
original_data = pd.read_csv('user_cognitive_styles.csv')
kmeans_data = pd.read_csv('KMeans Clusters.csv')

# Ensuring the 'user_id' column in both dataframes is of the same type
original_data['user_id'] = original_data['user_id'].astype(float)
kmeans_data['user_id'] = kmeans_data['user_id'].astype(float)

# Mapping Abstract and Concrete to 1 and 0 in original data for comparison
mapping = {'Abstract': 1, 'Concrete': 0}
original_data['Cognitive_Style'] = original_data['Cognitive_Style'].map(mapping)

# Merging both datasets on user_id to compare the Cognitive_Style
merged_data = pd.merge(original_data, kmeans_data, on='user_id', suffixes=('_orig', '_kmeans'))

# Checking for matches between the original data and k-means classification
merged_data['match'] = merged_data['Cognitive_Style_orig'] == merged_data['Cognitive_Style_kmeans']

# Calculating the accuracy: proportion of correct matches
accuracy = merged_data['match'].mean()

# Print out the accuracy
print(f"Accuracy: {accuracy:.2%}")

#detailed comparison data
merged_data.to_csv('detailed_comparison_of_segmentations.csv', index=False)

Accuracy: 74.52%
